# Parameter-Efficient Fine-Tuning: LoRA Implementation
## Week 7 — Day 4 | DI GenAI & Machine Learning Bootcamp 2026

**Objectives:**
- Understand LoRA (Low-Rank Adaptation) — the core PEFT technique used to fine-tune LLMs cheaply
- Implement `LoRALayer` with low-rank matrices A and B from scratch
- Wrap existing `nn.Linear` layers with `LinearWithLoRA` and `LinearWithLoRAMerged`
- Build an MLP on MNIST and replace its layers with LoRA variants
- Freeze original weights so only the tiny LoRA parameters are trained

**Exercises:**
1. Implementing the `LoRALayer`
2. Implementing `LinearWithLoRA`
3. Applying LoRA to a simple linear layer
4. Merging LoRA matrices — `LinearWithLoRAMerged`
5. MLP with LoRA layers
6. Freeze original layers and train only LoRA parameters

> **Run on Google Colab** — GPU recommended.

In [ ]:
!pip install --quiet torch torchvision matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import time
import copy

torch.manual_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} ✓")
print(f"Device  : {DEVICE}")

## 🌟 Exercise 1 — Implementing the `LoRALayer`

### What is LoRA?

**Low-Rank Adaptation (LoRA)** is a PEFT technique that avoids updating the full weight matrix `W ∈ ℝ^(d×k)` during fine-tuning. Instead it learns a *low-rank decomposition* of the weight update `ΔW = α · A · B` where:

- `A ∈ ℝ^(d×r)` — initialized with a scaled Gaussian (breaks symmetry)
- `B ∈ ℝ^(r×k)` — initialized to **zeros** (so LoRA starts as an identity / no-op)
- `r` — rank, a small integer (4, 8, 16) — far fewer parameters than the full `d×k` matrix
- `α` — scaling factor that controls the LoRA contribution magnitude

**Parameter savings**: for `d=k=768, r=8`:  
- Original: 768 × 768 = **589,824** parameters  
- LoRA: 768×8 + 8×768 = **12,288** parameters ← **48× fewer**

The pretrained weights stay **frozen**; only A and B are updated during fine-tuning.

In [ ]:
class LoRALayer(nn.Module):
    """
    Low-rank adaptation: computes  alpha * (x @ A @ B)
    A is initialized with scaled Gaussian noise; B with zeros.
    """

    def __init__(self, in_dim: int, out_dim: int, rank: int, alpha: float):
        super().__init__()
        std_dev = 1 / torch.sqrt(torch.tensor(rank).float())
        self.A     = nn.Parameter(torch.randn(in_dim, rank) * std_dev)
        self.B     = nn.Parameter(torch.zeros(rank, out_dim))
        self.alpha = alpha

    def forward(self, x):
        # x: (..., in_dim)  →  (..., out_dim)
        x = self.alpha * (x @ self.A @ self.B)
        return x


# ── Shape & sanity checks ─────────────────────────────────────────────────────
torch.manual_seed(42)
IN, OUT, RANK, ALPHA = 10, 6, 4, 8

lora = LoRALayer(IN, OUT, RANK, ALPHA)
x_test = torch.randn(3, IN)   # batch of 3

out_lora = lora(x_test)

print(f"LoRALayer — shape check")
print(f"  in_dim={IN}  out_dim={OUT}  rank={RANK}  alpha={ALPHA}")
print(f"  A shape : {tuple(lora.A.shape)}  (in_dim × rank)")
print(f"  B shape : {tuple(lora.B.shape)}  (rank × out_dim)")
print(f"  x  shape: {tuple(x_test.shape)}")
print(f"  out shape: {tuple(out_lora.shape)}  ✓")
print(f"\n  B is zeros at init → output should be all-zero:")
print(f"  max|out| = {out_lora.abs().max().item():.6f}  (expect 0.0)")

lora_params = sum(p.numel() for p in lora.parameters())
full_params  = IN * OUT
print(f"\n  LoRA params : {lora_params}  ({IN}×{RANK} + {RANK}×{OUT})")
print(f"  Full params : {full_params}  ({IN}×{OUT})")
print(f"  Savings     : {full_params - lora_params} fewer params ({100*(1-lora_params/full_params):.0f}% reduction)")

## 🌟 Exercise 2 — Implementing `LinearWithLoRA`

`LinearWithLoRA` wraps an existing frozen `nn.Linear` and adds a parallel `LoRALayer` branch.  
At inference the output is simply `linear(x) + lora(x)` — no architectural changes required.

In [ ]:
class LinearWithLoRA(nn.Module):
    """
    Wraps a frozen nn.Linear and adds a LoRALayer in parallel.
    forward(x) = linear(x) + lora(x)
    """

    def __init__(self, linear: nn.Linear, rank: int, alpha: float):
        super().__init__()
        self.linear = linear
        self.lora   = LoRALayer(
            linear.in_features, linear.out_features, rank, alpha
        )

    def forward(self, x):
        return self.linear(x) + self.lora(x)


# ── Test ──────────────────────────────────────────────────────────────────────
random_seed = 123
torch.manual_seed(random_seed)

layer = nn.Linear(5, 3)
x     = torch.randn(1, 5)

print(f"Original nn.Linear layer  : {layer}")
print(f"Input x                   : {x}")
print(f"Original output           : {layer(x)}")

# Wrap with LoRA
layer_lora_1 = LinearWithLoRA(layer, rank=4, alpha=8)
print(f"\nLinearWithLoRA output     : {layer_lora_1(x)}")
print(f"(same as original at init — B=0 so LoRA adds nothing yet)")

total_orig = sum(p.numel() for p in layer.parameters())
total_lora = sum(p.numel() for p in layer_lora_1.parameters())
print(f"\nOriginal params  : {total_orig}")
print(f"LoRA-wrapped params total: {total_lora}  (+{total_lora - total_orig} from A & B)")

## 🌟 Exercise 3 — Applying LoRA to a Simple Network

Replacing a plain `nn.Linear` with `LinearWithLoRA` should produce **identical output at initialisation** (because B=0), which we verify explicitly here.

In [ ]:
torch.manual_seed(random_seed)

# Simple single-layer network
single_layer = nn.Linear(10, 5)
x_single     = torch.randn(2, 10)

orig_out = single_layer(x_single)
print(f"Single nn.Linear output:\n{orig_out}\n")

# Replace with LinearWithLoRA
single_layer_lora = LinearWithLoRA(single_layer, rank=4, alpha=8)
lora_out = single_layer_lora(x_single)

print(f"LinearWithLoRA output (at init):\n{lora_out}\n")

# Verify outputs are identical (B=0 → LoRA adds nothing)
max_diff = (orig_out - lora_out).abs().max().item()
print(f"Max absolute difference between outputs : {max_diff:.8f}  (should be 0.0)")
assert max_diff < 1e-6, "Outputs should be identical at initialisation!"
print("✓ Outputs are identical at initialisation — LoRA is a no-op when B=0")

## 🌟 Exercise 4 — `LinearWithLoRAMerged`: Merging Weights for Efficiency

`LinearWithLoRA` performs **two separate matrix multiplications** at inference (`linear(x)` and `lora(x)`). `LinearWithLoRAMerged` **pre-merges** the LoRA update into a single combined weight matrix before calling `F.linear`, so inference is as fast as a standard linear layer with no extra cost.

In [ ]:
class LinearWithLoRAMerged(nn.Module):
    """
    Equivalent to LinearWithLoRA but merges A·B into the weight matrix
    before the matrix multiply → single F.linear call at inference.

    combined_weight = W + alpha * (A @ B)^T
    output = F.linear(x, combined_weight, bias)
    """

    def __init__(self, linear: nn.Linear, rank: int, alpha: float):
        super().__init__()
        self.linear = linear
        self.lora   = LoRALayer(
            linear.in_features, linear.out_features, rank, alpha
        )

    def forward(self, x):
        lora_update     = self.lora.A @ self.lora.B          # (in, rank) @ (rank, out) → (in, out)
        combined_weight = self.linear.weight + self.lora.alpha * lora_update.T
        return F.linear(x, combined_weight, self.linear.bias)


# ── Verify equivalence with LinearWithLoRA ────────────────────────────────────
torch.manual_seed(random_seed)
test_linear = nn.Linear(5, 3)
x_eq        = torch.randn(2, 5)

lora1 = LinearWithLoRA(test_linear, rank=4, alpha=8)
lora2 = LinearWithLoRAMerged(test_linear, rank=4, alpha=8)

# Copy the same A and B into both so outputs are truly comparable
with torch.no_grad():
    lora2.lora.A.copy_(lora1.lora.A)
    lora2.lora.B.copy_(lora1.lora.B)

out1 = lora1(x_eq)
out2 = lora2(x_eq)

print("LinearWithLoRA output        :", out1)
print("LinearWithLoRAMerged output  :", out2)
max_diff = (out1 - out2).abs().max().item()
print(f"\nMax absolute difference : {max_diff:.2e}  (should be ≈0)")
assert max_diff < 1e-5, "Merged and unmerged outputs should be numerically identical!"
print("✓ Outputs match — merged approach is numerically equivalent")

## 🌟 Exercise 5 — MLP with LoRA Layers

We build a 3-layer MLP, pretrain it on MNIST, then deep-copy the model and replace every `nn.Linear` with `LinearWithLoRAMerged` to demonstrate that accuracy is preserved before LoRA training begins.

In [ ]:
class MultilayerPerceptron(nn.Module):
    """3-layer fully connected network: input → hidden1 → hidden2 → output."""

    def __init__(self, num_features: int, num_hidden_1: int,
                 num_hidden_2: int, num_classes: int):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(num_features, num_hidden_1),   # layer 0
            nn.ReLU(),                                # layer 1
            nn.Linear(num_hidden_1, num_hidden_2),   # layer 2
            nn.ReLU(),                                # layer 3
            nn.Linear(num_hidden_2, num_classes),    # layer 4
        )

    def forward(self, x):
        return self.layers(x)


# ── Hyperparameters ───────────────────────────────────────────────────────────
NUM_FEATURES = 784    # 28 × 28 flattened MNIST pixels
NUM_HIDDEN_1 = 128
NUM_HIDDEN_2 = 256
NUM_CLASSES  = 10

LEARNING_RATE = 0.001
NUM_EPOCHS    = 2
BATCH_SIZE    = 64

model = MultilayerPerceptron(
    num_features = NUM_FEATURES,
    num_hidden_1 = NUM_HIDDEN_1,
    num_hidden_2 = NUM_HIDDEN_2,
    num_classes  = NUM_CLASSES,
).to(DEVICE)

optimizer_pretrained = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"Device    : {DEVICE}")
print(f"\nModel architecture:")
print(model)
n_total = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters : {n_total:,}")

In [ ]:
# ── MNIST DataLoaders ─────────────────────────────────────────────────────────
train_dataset = datasets.MNIST(
    root="data", train=True,
    transform=transforms.ToTensor(), download=True
)
test_dataset = datasets.MNIST(
    root="data", train=False,
    transform=transforms.ToTensor(), download=True
)

train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(dataset=test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

for images, labels in train_loader:
    print(f"Image batch dimensions : {images.shape}")
    print(f"Label batch dimensions : {labels.shape}")
    break

In [ ]:
def compute_accuracy(model, data_loader, device):
    model.eval()
    correct_pred, num_examples = 0, 0
    with torch.no_grad():
        for features, targets in data_loader:
            features       = features.view(-1, NUM_FEATURES).to(device)
            targets        = targets.to(device)
            logits         = model(features)
            _, pred_labels = torch.max(logits, 1)
            num_examples  += targets.size(0)
            correct_pred  += (pred_labels == targets).sum()
    return correct_pred.float() / num_examples * 100


def train(num_epochs, model, optimizer, train_loader, device):
    train_accs = []
    start_time = time.time()
    for epoch in range(num_epochs):
        model.train()
        for batch_idx, (features, targets) in enumerate(train_loader):
            features = features.view(-1, NUM_FEATURES).to(device)
            targets  = targets.to(device)

            logits = model(features)
            loss   = F.cross_entropy(logits, targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if not batch_idx % 400:
                print(f"Epoch: {epoch+1:03d}/{num_epochs:03d} | "
                      f"Batch {batch_idx:03d}/{len(train_loader):03d} | "
                      f"Loss: {loss.item():.4f}")

        acc = compute_accuracy(model, train_loader, device)
        train_accs.append(acc.item())
        with torch.set_grad_enabled(False):
            print(f"Epoch: {epoch+1:03d}/{num_epochs:03d} "
                  f"training accuracy: {acc:.2f}%")
        print(f"Time elapsed: {(time.time()-start_time)/60:.2f} min")

    print(f"Total Training Time: {(time.time()-start_time)/60:.2f} min")
    return train_accs


# ── Pretrain the base MLP ─────────────────────────────────────────────────────
print("=== Pretraining base MLP on MNIST ===")
train_accs_base = train(NUM_EPOCHS, model, optimizer_pretrained, train_loader, DEVICE)
test_acc_base   = compute_accuracy(model, test_loader, DEVICE)
print(f"\nTest accuracy (base MLP) : {test_acc_base:.2f}%")

In [ ]:
# ── Replace Linear layers with LinearWithLoRAMerged ──────────────────────────
model_lora = copy.deepcopy(model)

LORA_RANK  = 4
LORA_ALPHA = 8

model_lora.layers[0] = LinearWithLoRAMerged(model_lora.layers[0], rank=LORA_RANK, alpha=LORA_ALPHA)
model_lora.layers[2] = LinearWithLoRAMerged(model_lora.layers[2], rank=LORA_RANK, alpha=LORA_ALPHA)
model_lora.layers[4] = LinearWithLoRAMerged(model_lora.layers[4], rank=LORA_RANK, alpha=LORA_ALPHA)

model_lora.to(DEVICE)

print("MLP with LoRA layers:")
print(model_lora)

n_lora_total = sum(p.numel() for p in model_lora.parameters())
n_lora_only  = sum(p.numel() for n, p in model_lora.named_parameters()
                   if "lora" in n)
print(f"\nTotal params (with LoRA) : {n_lora_total:,}")
print(f"LoRA-only params         : {n_lora_only:,}  "
      f"({100*n_lora_only/n_lora_total:.1f}% of total)")

# Verify accuracy is preserved right after wrapping (B=0 → no change)
acc_orig  = compute_accuracy(model,      test_loader, DEVICE)
acc_lora0 = compute_accuracy(model_lora, test_loader, DEVICE)
print(f"\nTest accuracy — original MLP              : {acc_orig:.2f}%")
print(f"Test accuracy — LoRA-wrapped (B=0 init)   : {acc_lora0:.2f}%")
print("(should be identical ✓)")

## 🌟 Exercise 6 — Freeze Original Layers & Train Only LoRA

We freeze every `nn.Linear` inside the model so its weights are not updated during backprop. The `LoRALayer` parameters (A and B) are **not** `nn.Linear`, so they remain trainable — this is the core PEFT mechanism.

In [ ]:
def freeze_linear_layers(model):
    """Recursively freeze all nn.Linear parameters in the model."""
    for child in model.children():
        if isinstance(child, nn.Linear):
            for param in child.parameters():
                param.requires_grad = False
        else:
            freeze_linear_layers(child)   # recurse into sub-modules


freeze_linear_layers(model_lora)

print("Parameter trainability after freezing:")
print(f"  {'Parameter name':<40} {'Trainable':>10}  {'Shape'}")
print(f"  {'─'*65}")
for name, param in model_lora.named_parameters():
    status = "✓ True" if param.requires_grad else "✗ False"
    print(f"  {name:<40} {status:>10}  {tuple(param.shape)}")

n_trainable = sum(p.numel() for p in model_lora.parameters() if p.requires_grad)
n_frozen    = sum(p.numel() for p in model_lora.parameters() if not p.requires_grad)
print(f"\nTrainable params : {n_trainable:,}   (LoRA A + B only)")
print(f"Frozen params    : {n_frozen:,}   (original Linear W + bias)")
print(f"Training only {100*n_trainable/(n_trainable+n_frozen):.2f}% of all parameters")

In [ ]:
# ── Fine-tune only the LoRA parameters ───────────────────────────────────────
optimizer_lora = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model_lora.parameters()),
    lr=LEARNING_RATE
)

print("=== Fine-tuning: only LoRA layers trainable ===")
train_accs_lora = train(NUM_EPOCHS, model_lora, optimizer_lora, train_loader, DEVICE)
test_acc_lora   = compute_accuracy(model_lora, test_loader, DEVICE)

# ── Final comparison ──────────────────────────────────────────────────────────
print(f"\n{'─'*50}")
print(f"Test accuracy — base MLP (full training)   : {test_acc_base:.2f}%")
print(f"Test accuracy — LoRA fine-tune (frozen W)  : {test_acc_lora:.2f}%")
print(f"{'─'*50}")

# ── Accuracy curves ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
epochs_r = range(1, NUM_EPOCHS + 1)
ax.plot(epochs_r, train_accs_base, "b-o", label="Base MLP (all params)")
ax.plot(epochs_r, train_accs_lora, "r-o", label=f"LoRA fine-tune (rank={LORA_RANK})")
ax.set_xlabel("Epoch")
ax.set_ylabel("Training Accuracy (%)")
ax.set_title("Base MLP vs LoRA Fine-Tuning — MNIST", fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("lora_training_comparison.png", dpi=120, bbox_inches="tight")
plt.show()
print("Plot saved ✓")

In [ ]:
# ── Parameter efficiency summary ──────────────────────────────────────────────
print("Parameter Efficiency Summary")
print(f"{'─'*55}")
print(f"{'Component':<30} {'Params':>10}  {'Trainable'}")
print(f"{'─'*55}")

for name, param in model_lora.named_parameters():
    trainable = "✓" if param.requires_grad else "✗ (frozen)"
    print(f"  {name:<28} {param.numel():>10,}  {trainable}")

print(f"{'─'*55}")
print(f"  {'TOTAL':<28} {n_trainable + n_frozen:>10,}")
print(f"  {'TRAINABLE (LoRA only)':<28} {n_trainable:>10,}  ← {100*n_trainable/(n_trainable+n_frozen):.2f}%")
print(f"  {'FROZEN (original W)':<28} {n_frozen:>10,}")
print(f"\n✓ Only LoRA A and B matrices are updated during fine-tuning")

## Summary & Key Takeaways

| Exercise | Concept | Key point |
|---|---|---|
| 1 | `LoRALayer` | A: scaled Gaussian init; B: zeros init → LoRA is a no-op at start; `alpha * (x @ A @ B)` |
| 2 | `LinearWithLoRA` | Wraps `nn.Linear`; `forward = linear(x) + lora(x)` — two separate multiplications |
| 3 | Output equivalence | B=0 → LoRA adds exactly 0 → identical output to original layer at initialisation |
| 4 | `LinearWithLoRAMerged` | Pre-merges: `W_combined = W + α·(A@B)^T` → single `F.linear` call, no extra inference cost |
| 5 | MLP + LoRA | `layers[0,2,4]` replaced with `LinearWithLoRAMerged`; accuracy preserved before LoRA training |
| 6 | Freeze + fine-tune | `freeze_linear_layers` sets `param.requires_grad=False` on all `nn.Linear`; only LoRA A/B update |

### Why LoRA Works in Practice

- **Hypothesis**: weight updates during fine-tuning have low intrinsic rank — the full `ΔW` update lives in a low-dimensional subspace
- **Evidence**: LoRA achieves near-identical quality to full fine-tuning on BERT/GPT-2/LLaMA with rank as low as 4–16
- **Memory win**: only A and B stored in the optimizer state — crucial when the frozen W is billions of parameters
- **Merge trick**: `W_merged = W + α·(A@B)^T` → zero inference overhead; can be done once at deployment